# Solving the Steady-State 2D Radiative Transfer Equation (RTE) using an Integral-Form PINN

In this notebook, we solve the steady-state 2D Radiative Transfer Equation (RTE) in a participating square medium using a **PINN on the integral (Peierls) form**: the network approximates the smooth incident radiation $G(x, y)$ directly, instead of the 4D intensity $I(x, y, \mu, \eta)$.

### Case 2 (2D)
We focus on a 2D space + 3D direction test case with the following configuration:
* **Geometry:** $0 \le x \le L_x$ and $0 \le y \le L_y$ with $L_x = L_y = 1.0\text{ m}$ (Square medium).
* **Equation (integral form):**
  $$G(\vec r) = G_0(\vec r) + \int_{4\pi}\!\!\int_0^{p_b} e^{-\sigma p/\sin\theta}\,\frac{\sigma}{4\pi}\,G(\vec r - p\,\hat d)\,\frac{dp}{\sin\theta}\,d\Omega$$
  where $G_0$ is the uncollided contribution of the hot wall and the residual is $R = G_\theta - G_0 - \mathcal{K}[G_\theta]$.
* **Physical Properties:**
  * Scattering coefficient: $\sigma = 1.0\text{ m}^{-1}$ (Isotropic scattering).
  * Absorption coefficient: $\kappa = 0\text{ m}^{-1}$ (Non-absorbing scattering medium).
* **Boundary Conditions:** diffuse radiation $I = 1$ entering through the top wall ($y = 1$); the three other walls are cold ($I = 0$ incoming).

The integral operator $\mathcal{K}$ is **precomputed once** (backward-ray geometry); during training only $G_\theta$ at the ray points is re-evaluated, so there is **no autograd** in the loss. Validation as always: the in-notebook DOM reference and the exact anchor $G(0.5, 0.5) = \pi$.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. PINN Architecture Definition
Definition of the standard Neural Network (Multi-Layer Perceptron) to approximate the incident radiation $G(x, y)$ — 2 inputs.

In [ ]:
class PinnRFEEqG(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=4):
        super().__init__()
        layers = []
        layers.append(nn.Linear(2, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 2. Ray-Direction Quadrature for the Unit Sphere
Setting up the quadrature over solid angle used by the integral operator: Gauss-Legendre nodes for $\xi = \cos\theta$ on $[0, 1]$ (both hemispheres) and a uniform midpoint rule in $\phi$. The weights sum to $4\pi$.

In [ ]:
def init_quadrature(N_theta=10, N_phi=32):
    nodes_xi, weights_xi = np.polynomial.legendre.leggauss(N_theta)
    nodes_xi = 0.5 * (nodes_xi + 1.0)                 # xi in [0, 1] (weights not halved -> both hemispheres)
    nodes_phi = (np.arange(N_phi) + 0.5) * 2.0 * np.pi / N_phi

    quad_dirs = []
    for i in range(N_theta):
        sin_theta = np.sqrt(1.0 - nodes_xi[i] ** 2)
        for phi in nodes_phi:
            mu = sin_theta * np.cos(phi)
            eta = sin_theta * np.sin(phi)
            w = weights_xi[i] * (2.0 * np.pi / N_phi)
            quad_dirs.append((mu, eta, sin_theta, w))
    return np.array(quad_dirs)

## 3. Collocation Points and Integral-Operator Precomputation
Random collocation points in the square, and for each point + each ray direction: the backward ray to the boundary, the uncollided contribution $G_0$ (if the ray exits through the hot wall), and the attenuated in-scattering coefficients along the ray (trapezoidal rule, `N_t` steps). Computed **once**.

In [ ]:
def generer_points_collocation(n_pde):
    return np.random.rand(n_pde, 2) * 0.98 + 0.01

def dist_to_boundary(rx, ry, dir_x, dir_y):
    cands = []
    if dir_x >  1e-12: cands.append((rx / dir_x, 2))
    if dir_x < -1e-12: cands.append(((rx - 1.0) / dir_x, 3))
    if dir_y >  1e-12: cands.append((ry / dir_y, 0))
    if dir_y < -1e-12: cands.append(((ry - 1.0) / dir_y, 1))
    cands = [(p, wall) for p, wall in cands if p > 1e-9]
    return min(cands) if cands else (0.0, -1)

def construire_operateur(pts, quad_dirs, sigma, N_t):
    n = pts.shape[0]
    Q = quad_dirs.shape[0]
    QT = Q * N_t

    G_uncollided = np.zeros(n)
    ray_points = np.zeros((n, QT, 2), np.float32)
    ray_coeffs = np.zeros((n, QT), np.float32)

    for i in range(n):
        rx, ry = pts[i]
        for q, (mu, eta, sin_theta, w) in enumerate(quad_dirs):
            dir_x, dir_y = mu / sin_theta, eta / sin_theta
            p_bord, wall = dist_to_boundary(rx, ry, dir_x, dir_y)
            sl = slice(q * N_t, (q + 1) * N_t)
            if p_bord <= 0:
                ray_points[i, sl, 0] = rx
                ray_points[i, sl, 1] = ry
                continue
            if wall == 1:                              # la paroi haute (chaude) : contribution non-collisionnee
                G_uncollided[i] += w * np.exp(-sigma * p_bord / sin_theta)
            ps = np.linspace(0.0, p_bord, N_t)
            dp = ps[1] - ps[0]
            trap = np.full(N_t, dp); trap[0] *= 0.5; trap[-1] *= 0.5
            ray_points[i, sl, 0] = rx - ps * dir_x
            ray_points[i, sl, 1] = ry - ps * dir_y
            ray_coeffs[i, sl] = w * np.exp(-sigma * ps / sin_theta) * (sigma / (4.0 * np.pi)) / sin_theta * trap

    return G_uncollided, ray_points, ray_coeffs

## 4. Loss Function Definition
Residual of the integral equation, $R = G_\theta - G_0 - \mathcal{K}[G_\theta]$, on a batch of collocation points. **No `torch.autograd.grad`** — the network is only evaluated at the collocation and ray points.

In [ ]:
def calc_residual_loss(model, idx, pts_colloc, ray_points, ray_coeffs, G_uncollided, QT):
    G_rays = model(ray_points[idx].reshape(-1, 2)).view(idx.shape[0], QT)
    K_G = torch.sum(ray_coeffs[idx] * G_rays, dim=1, keepdim=True)
    G_pts = model(pts_colloc[idx])
    residual = G_pts - G_uncollided[idx] - K_G
    return torch.mean(residual ** 2)

## 5. Hardware (Device), Model, and Optimizer Initialization
Hardware detection, quadrature and operator precomputation, model creation and data transfer to the device.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

np.random.seed(0)
torch.manual_seed(0)

n_pde = 3000
N_theta = 10
N_phi = 32
N_t = 48
batch_size = 64
kappa = 0.0
sigma = 1.0

quad_dirs = init_quadrature(N_theta=N_theta, N_phi=N_phi)
Q = quad_dirs.shape[0]
QT = Q * N_t
print(f"{Q} ray directions, {QT} ray points per collocation point")

pts_np = generer_points_collocation(n_pde)
G_uncollided_np, ray_points_np, ray_coeffs_np = construire_operateur(pts_np, quad_dirs, sigma, N_t)
print(f"operateur precalcule (stockage {ray_points_np.nbytes / 1e6:.0f} MB)")

pts_colloc = torch.tensor(pts_np, dtype=torch.float32).to(device)
ray_points = torch.tensor(ray_points_np, dtype=torch.float32).to(device)
ray_coeffs = torch.tensor(ray_coeffs_np, dtype=torch.float32).to(device)
G_uncollided = torch.tensor(G_uncollided_np, dtype=torch.float32).view(-1, 1).to(device)

modele = PinnRFEEqG(hidden_dim=64, num_layers=4).to(device)

## 6. Model Training (Adam)
Training phase using the Adam optimizer, on random mini-batches of collocation points (fits in 8 GB of GPU memory), with a stepwise learning-rate decay.

In [ ]:
optimizer = optim.Adam(modele.parameters(), lr=2e-3)
epochs = 20000

for epoch in range(epochs):
    for g in optimizer.param_groups:                 # decay 2e-3 -> 2.5e-4
        g['lr'] = 2e-3 * (0.5 ** (epoch // 5000))
    optimizer.zero_grad()
    idx = torch.randint(0, n_pde, (batch_size,), device=device)
    loss = calc_residual_loss(modele, idx, pts_colloc, ray_points, ray_coeffs, G_uncollided, QT)
    loss.backward()
    optimizer.step()

    if epoch % 1000 == 0:
        print(f"Epoque {epoch:05d} | Loss: {loss.item():.2e}")

## 7. Model Training (L-BFGS)
Fine-tuning with L-BFGS on a **fixed** subset of collocation points (L-BFGS requires a deterministic loss; the subset is kept small so the operator graph fits in memory).

In [ ]:
idx_fix = torch.arange(0, min(128, n_pde), device=device)

def closure():
    optimizer_lbfgs.zero_grad()
    loss = calc_residual_loss(modele, idx_fix, pts_colloc, ray_points, ray_coeffs, G_uncollided, QT)
    loss.backward()
    return loss

optimizer_lbfgs = optim.LBFGS(modele.parameters(), line_search_fn="strong_wolfe", max_iter=20)
lbfgs_epochs = 200

for epoch in range(lbfgs_epochs):
    loss = optimizer_lbfgs.step(closure)
    if epoch % 20 == 0:
        print(f"Epoque LBFGS {epoch:03d} | Loss: {loss.item():.2e}")

## 8. Visualizing the Results and Validation against a 2D DOM Reference

To validate the PINN solution, we compare the predicted incident radiation $G(x, y)$ along the centerlines against a reference computed by a **Discrete Ordinates (DOM) solver** — same equation and boundary conditions (implicit upwind sweeps + source iteration, grid-converged).

### Exact analytical anchor
For a purely scattering square ($\omega = 1$) with a single diffuse wall at $I = 1$ and cold black walls, superposing the four 90°-rotations of the problem yields $I \equiv 1$, hence **exactly** $G(0.5, 0.5) = \pi \approx 3.1416$.

In [ ]:
# Solveur DOM 2D de reference : meme equation et memes BCs que le PINN.
# Schema upwind implicite, balayages vectorises par fronts d'onde (les cellules
# d'une anti-diagonale ne dependent que de la precedente), iteration de la source.

def resoudre_dom(M=201, N_xi=12, N_phi=48, beta=1.0, sigma_dom=1.0, tol=1e-8, max_iter=2000):
    dx = 1.0 / (M - 1)

    nx, wx = np.polynomial.legendre.leggauss(N_xi)
    xi = 0.5 * (nx + 1.0)
    phi = (np.arange(N_phi) + 0.5) * 2.0 * np.pi / N_phi
    XI, PHI = np.meshgrid(xi, phi, indexing='ij')
    W = np.outer(wx, np.full(N_phi, 2.0 * np.pi / N_phi)).ravel()
    st = np.sqrt(1.0 - XI**2)
    MU = (st * np.cos(PHI)).ravel()
    ETA = (st * np.sin(PHI)).ravel()

    quadrants = []
    for smu in (1, -1):
        for seta in (1, -1):
            sel = (np.sign(MU) == smu) & (np.sign(ETA) == seta)
            quadrants.append((smu, seta, MU[sel], ETA[sel], W[sel]))

    def balayage(smu, seta, mu, eta, S):
        a = np.abs(mu)[:, None] / dx
        b = np.abs(eta)[:, None] / dx
        I = np.zeros((mu.size, M, M))
        bc_y = 1.0 if seta < 0 else 0.0   # paroi haute (y=1) chaude, les autres a 0
        Sf = S[::smu, ::seta]             # vues retournees : balayage toujours croissant
        If = I[:, ::smu, ::seta]
        denom = a + b + beta
        for d in range(2 * M - 1):
            ii = np.arange(max(0, d - M + 1), min(d, M - 1) + 1)
            jj = d - ii
            Ix = If[:, np.where(ii > 0, ii - 1, 0), jj]
            Ix[:, ii == 0] = 0.0
            Iy = If[:, ii, np.where(jj > 0, jj - 1, 0)]
            Iy[:, jj == 0] = bc_y
            If[:, ii, jj] = (a * Ix + b * Iy + Sf[ii, jj][None, :]) / denom
        return I

    G = np.zeros((M, M))
    for it in range(max_iter):
        S = (sigma_dom / (4.0 * np.pi)) * G
        Gn = np.zeros_like(G)
        for smu, seta, mu, eta, w in quadrants:
            Gn += np.tensordot(w, balayage(smu, seta, mu, eta, S), axes=(0, 0))
        diff = np.max(np.abs(Gn - G))
        G = Gn
        if diff < tol:
            break
    print(f"DOM converge en {it} iterations (diff = {diff:.2e})")
    return G

M_dom = 201
G_dom = resoudre_dom(M=M_dom, beta=kappa + sigma, sigma_dom=sigma)
grid_dom = np.linspace(0.0, 1.0, M_dom)
mid_dom = (M_dom - 1) // 2
print(f"G(0.5, 0.5) DOM = {G_dom[mid_dom, mid_dom]:.4f}   (exact = pi = {np.pi:.4f})")

In [ ]:
def evaluer_G(model, x_grid, y_grid, device):
    X, Y = np.meshgrid(x_grid, y_grid)
    inp = torch.tensor(np.stack([X.ravel(), Y.ravel()], axis=1), dtype=torch.float32).to(device)
    with torch.no_grad():
        G = model(inp).cpu().numpy().reshape(Y.shape)
    return G

x_vals = np.linspace(0.0, 1.0, 100)
y_vals = np.linspace(0.0, 1.0, 100)
G_pred = evaluer_G(modele, x_vals, y_vals, device)

# Plot Heatmap of G(x, y)
fig, ax = plt.subplots(figsize=(7, 6))
X, Y = np.meshgrid(x_vals, y_vals)
im = ax.pcolormesh(X, Y, G_pred, cmap='jet', shading='auto')
ax.set_title("Incident Radiation $G(x, y)$ (2D PINN, integral form)")
ax.set_xlabel("Position $x$")
ax.set_ylabel("Position $y$")
fig.colorbar(im, label="Incident Radiation $G$")
plt.tight_layout()
fig.savefig("intensity_heatmap_cas2_integral.png", dpi=150)
plt.show()

# Plot Line Profiles vs DOM reference
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Line x = 0.5
y_vals_line = np.linspace(0.0, 1.0, 100)
G_x05 = evaluer_G(modele, np.array([0.5]), y_vals_line, device).flatten()
axes[0].plot(y_vals_line, G_x05, color='red', linewidth=2, label="PINN")
axes[0].plot(grid_dom, G_dom[mid_dom, :], 'k--', linewidth=1.5, label="DOM (référence)")
axes[0].plot(0.5, np.pi, 'b*', markersize=12, label="Exact : $G(0.5, 0.5) = \\pi$")
axes[0].set_title("Incident Radiation $G(0.5, y)$")
axes[0].set_xlabel("Position $y$")
axes[0].set_ylabel("$G(0.5, y)$")
axes[0].grid(True)
axes[0].legend()

# Line y = 0.5
x_vals_line = np.linspace(0.0, 1.0, 100)
G_y05 = evaluer_G(modele, x_vals_line, np.array([0.5]), device).flatten()
axes[1].plot(x_vals_line, G_y05, color='blue', linewidth=2, label="PINN")
axes[1].plot(grid_dom, G_dom[:, mid_dom], 'k--', linewidth=1.5, label="DOM (référence)")
axes[1].plot(0.5, np.pi, 'b*', markersize=12, label="Exact : $G(0.5, 0.5) = \\pi$")
axes[1].set_title("Incident Radiation $G(x, 0.5)$")
axes[1].set_xlabel("Position $x$")
axes[1].set_ylabel("$G(x, 0.5)$")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
fig.savefig("intensity_boundaries_cas2_integral.png", dpi=150)
plt.show()

# Erreur relative PINN vs DOM sur la ligne verticale centrale
ref_y = np.linspace(0.0, 1.0, 11)
ref_G_vertical = np.interp(ref_y, grid_dom, G_dom[mid_dom, :])
G_pinn_at_ref = np.interp(ref_y, y_vals_line, G_x05)
err = np.abs(G_pinn_at_ref - ref_G_vertical) / np.abs(ref_G_vertical)
print("y            :", np.round(ref_y, 2))
print("G DOM        :", np.round(ref_G_vertical, 3))
print("G PINN       :", np.round(G_pinn_at_ref, 3))
print("erreur rel.  :", np.round(100 * err, 1), "%")